SPDX-FileCopyrightText: © 2026 Idiap Research Institute <contact@idiap.ch>
SPDX-FileContributor: David Alonso del Barrio <ddbarrio@idiap.ch>
SPDX-License-Identifier: GPL-3.0-only

# EMN Factsheet Harmonization Pipeline 




In [1]:
import pandas as pd
import os
import re
import glob
# Define the data structure for harmonization
base_dir = './data'
search_path = os.path.join(base_dir, '*', 'final', '*.csv')
csv_files = glob.glob(search_path)

## 1. Data Collection
We collect all cleaned CSV files

In [2]:
dataframes = []

for file_path in csv_files:
    try:
        df = pd.read_csv(file_path)
        dataframes.append(df)
    except Exception as e:
        print(f"Could not load {file_path}: {e}")

if dataframes:
    combined_df = pd.concat(dataframes, ignore_index=True)
    print(f"Successfully combined {len(dataframes)} files into one dataset.")
else:
    print("No files found! Check your folder structure.")

Successfully combined 2 files into one dataset.


## 2. Text Extraction
We apply a regex-based parser to identify main sections and subsections within the reports. This logic maps heterogeneous text structures into a standard dictionary format.

In [3]:
import re
from collections import defaultdict

# List of all possible main titles and their variations
main_titles = [
    "KEY POINTS", "OVERARCHING CHANGES", "OVERARCHING AND CROSS-CUTTING CHANGES","COMMON EUROPEAN ASYLUM SYSTEM",
    "LEGAL MIGRATION AND MOBILITY", "LEGAL MIGRATION", "TEMPORARY PROTECTION","EUROPEAN POLICY ON LEGAL MIGRATION AND INTEGRATION",
    "TEMPORARY PROTECTION AND OTHER MEASURES IN RESPONSE TO RUSSIA'S WAR OF AGGRESSION AGAINST UKRAINE",
    "TEMPORARY PROTECTION AND OTHER MEASURES IN RESPONSE TO RUSSIA\’S WAR OF AGGRESSION AGAINST UKRAINE",
    "TEMPORARY PROTECTION AND OTHER MEASURES IN RESPONSE TO RUSSIA’S WAR OF AGGRESSION AGAINST UKRAINE",
    "INTERNATIONAL PROTECTION", "INTERNATIONAL PROTECTION INCLUDING ASYLUM","INTERNATIONAL PROTECTION AND ASYLUM",
    'INTERNATIONAL PROTECTION INCLUDING ASYLUM RELOCATION AND RESETTLEMENT',
    "MINORS AND OTHER VULNERABLE GROUPS", "UNACCOMPANIED MINORS AND OTHER VULNERABLE GROUPS",
    "INTEGRATION", "INTEGRATION AND INCLUSION OF MIGRANTS", "CITIZENSHIP AND STATELESSNESS",
    "EXTERNAL DIMENSION OF EU MIGRATION POLICY",
    "BORDERS, SCHENGEN AND VISAS","BORDERS, SCHENGEN AND VISA", "BORDERS, SCHENGEN, AND VISAS", "BORDERS AND VISAS", "BORDERS, VISA AND SCHENGEN","BORDER MANAGEMENT AND VISA POLICIES ","SCHENGEN GOVERNANCE AND OTHER DEVELOPMENTS IN BORDER MANAGEMENT AND VISA POLICIES",
    "IRREGULAR MIGRATION INCLUDING MIGRANT SMUGGLING", "IRREGULAR MIGRATION","IRREGULAR MIGRATION AND RETURN",
    "IRREGULAR MIGRATION INCLUDING MIGRANT TRAFFICKING","HUMAN BEINGS’ TRAFFICKING",
    "TRAFFICKING IN HUMAN BEINGS","ACTIONS AGAINST TRAFFICKING IN HUMAN BEINGS", "RETURN AND READMISSION",'RETURN AND REINTEGRATION',
    "MIGRATION AND DEVELOPMENT","MAXIMISING DEVELOPMENT IMPACT OF MIGRATION AND MOBILITY"
]

# Expanded regex to include special subtitle patterns
main_title_pattern = re.compile(r'^(' + '|'.join(re.escape(title) for title in main_titles) + r')$')
subtitle_pattern = re.compile(r'^[A-Z][A-Z _/,();-]+$')

def parse_text(text):

    parsed_content = {}
    
    current_title = None
    current_subtitle = None

    for line in text.splitlines():
        line = line.strip()
        
        # Check if the line matches a main title
        if main_title_match := main_title_pattern.match(line):
            current_title = main_title_match.group(0)
            parsed_content[current_title] = defaultdict(list)
            current_subtitle = None
        
        # If the line matches the refined subtitle pattern and we have a current title
        elif current_title and subtitle_pattern.match(line):
            # Treat this as a subtitle if it's not a recognized main title
            if not main_title_pattern.match(line):
                current_subtitle = line
            else:
                # Reset subtitle as this is a new main title instead of subtitle
                current_subtitle = None

        # Otherwise, treat it as paragraph content for the current title/subtitle
        elif current_title:
            if current_subtitle:
                # Add content under the current subtitle
                parsed_content[current_title][current_subtitle].append(line)
            else:
                # Add content directly under the main title if no subtitle
                parsed_content[current_title].setdefault('Main', []).append(line)

    # Convert defaultdicts to regular dicts for output
    parsed_content = {title: dict(subs) for title, subs in parsed_content.items()}
    return parsed_content


combined_df['Parsed_Content'] = combined_df['text'].apply(parse_text)




In [4]:
records = []
for idx, row in combined_df.iterrows():
    for section, subsections in row['Parsed_Content'].items():
        # Handle cases where subsections might be a list or a dict
        if isinstance(subsections, dict):
            for sub, content in subsections.items():
                records.append({
                    'country': row['country'],
                    'year': row['year'],
                    'section': section,
                    'subsection': sub,
                    'content': " ".join(content) if isinstance(content, list) else content
                })
        else:
            records.append({
                'country': row['country'],
                'year': row['year'],
                'section': section,
                'subsection': None,
                'content': " ".join(subsections) if isinstance(subsections, list) else subsections
            })

# 2. Create the DataFrame and clean whitespace
new_df = pd.DataFrame(records)
new_df['content'] = new_df['content'].str.strip()

## 3. Thematic/Section Harmonization
To ensure longitudinal consistency, we mapped various thematic section titles used in EMN reports over the years (2012–2024) into a unified set of standardized categories. This normalization allows for meaningful temporal analysis across reports where terminology shifted over time. Please note that this dictionary is the one we have used to harmonize all years from 2012-2024, although in the repository we exemplify the entire process with the data from 2022 and 2018.

In [5]:
title_mapping = {
    'BORDERS AND VISAS': 'BORDERS AND VISAS',
    'BORDERS, SCHENGEN AND VISAS': 'BORDERS AND VISAS',
    'BORDERS, VISA AND SCHENGEN': 'BORDERS AND VISAS',
    "BORDERS, SCHENGEN, AND VISAS":'BORDERS AND VISAS',
    "BORDERS, SCHENGEN AND VISA":'BORDERS AND VISAS',
    "BORDER MANAGEMENT AND VISA POLICIES ":'BORDERS AND VISAS',
    "BORDER MANAGEMENT AND VISA POLICIES":'BORDERS AND VISAS',
    "SCHENGEN GOVERNANCE AND OTHER DEVELOPMENTS IN BORDER MANAGEMENT AND VISA POLICIES":'BORDERS AND VISAS',
    
    
    'INTEGRATION': 'INTEGRATION',
    'INTEGRATION AND INCLUSION OF MIGRANTS': 'INTEGRATION',
    
    'INTERNATIONAL PROTECTION': 'INTERNATIONAL PROTECTION AND ASYLUM',
    'INTERNATIONAL PROTECTION INCLUDING ASYLUM': 'INTERNATIONAL PROTECTION AND ASYLUM',
    'INTERNATIONAL PROTECTION INCLUDING ASYLUM RELOCATION AND RESETTLEMENT':'INTERNATIONAL PROTECTION AND ASYLUM',
    'COMMON EUROPEAN ASYLUM SYSTEM': 'INTERNATIONAL PROTECTION AND ASYLUM',
    'INTERNATIONAL PROTECTION AND ASYLUM': 'INTERNATIONAL PROTECTION AND ASYLUM',
    
    'IRREGULAR MIGRATION': 'IRREGULAR MIGRATION',
    'IRREGULAR MIGRATION INCLUDING MIGRANT SMUGGLING': 'IRREGULAR MIGRATION',
    'IRREGULAR MIGRATION AND RETURN': 'IRREGULAR MIGRATION',
    'IRREGULAR MIGRATION INCLUDING MIGRANT TRAFFICKING' : 'IRREGULAR MIGRATION',
    
    'LEGAL MIGRATION': 'LEGAL MIGRATION AND MOBILITY',
    'LEGAL MIGRATION AND MOBILITY': 'LEGAL MIGRATION AND MOBILITY',
    'EUROPEAN POLICY ON LEGAL MIGRATION AND INTEGRATION':'LEGAL MIGRATION AND MOBILITY',

    
    'MINORS AND OTHER VULNERABLE GROUPS': 'MINORS AND VULNERABLE GROUPS',
    'UNACCOMPANIED MINORS AND OTHER VULNERABLE GROUPS': 'MINORS AND VULNERABLE GROUPS',
    
    'OVERARCHING AND CROSS-CUTTING CHANGES': 'OVERARCHING CHANGES',
    'OVERARCHING CHANGES': 'OVERARCHING CHANGES',
    
    'TEMPORARY PROTECTION': 'TEMPORARY PROTECTION',
    'TEMPORARY PROTECTION AND OTHER MEASURES IN RESPONSE TO RUSSIA\'S WAR OF AGGRESSION AGAINST UKRAINE': 'TEMPORARY PROTECTION AND MEASURES IN RESPONSE TO AGGRESSION AGAINST UKRAINE',
    'TEMPORARY PROTECTION AND OTHER MEASURES IN RESPONSE TO RUSSIA’S WAR OF AGGRESSION AGAINST UKRAINE': 'TEMPORARY PROTECTION AND MEASURES IN RESPONSE TO AGGRESSION AGAINST UKRAINE',
    'RETURN AND REINTEGRATION':'RETURN AND READMISSION',
    'RETURN AND READMISSION': 'RETURN AND READMISSION',
    
    'TRAFFICKING IN HUMAN BEINGS': 'TRAFFICKING IN HUMAN BEINGS',
    'ACTIONS AGAINST TRAFFICKING IN HUMAN BEINGS':'TRAFFICKING IN HUMAN BEINGS',
    "HUMAN BEINGS’ TRAFFICKING":'TRAFFICKING IN HUMAN BEINGS',
    
    "MAXIMISING DEVELOPMENT IMPACT OF MIGRATION AND MOBILITY": 'MIGRATION AND DEVELOPMENT',
    'MIGRATION AND DEVELOPMENT': 'MIGRATION AND DEVELOPMENT',

    
    # Titles without grouping
    'KEY POINTS': 'KEY POINTS',
    'CITIZENSHIP AND STATELESSNESS': 'CITIZENSHIP AND STATELESSNESS',
    'EXTERNAL DIMENSION OF EU MIGRATION POLICY': 'EXTERNAL DIMENSION OF EU MIGRATION POLICY',  
}

In [6]:
new_df['grouped_title_section'] = new_df['section'].map(title_mapping)
unmapped = new_df[new_df['grouped_title_section'].isna()]['section'].unique()
if len(unmapped) > 0:
    print(f"Warning: The following sections were not mapped: {unmapped}")
else:
    print("All sections successfully mapped.")

new_df


All sections successfully mapped.


,country,year,section,subsection,content,grouped_title_section
0,Austria,2018,KEY POINTS,Main,The Act Amending the Aliens Law 2018 was adopt...,KEY POINTS
1,Austria,2018,LEGAL MIGRATION AND MOBILITY,PROMOTING LEGAL MIGRATION CHANNELS,The Act Amending the Aliens Law 2018 adopted l...,LEGAL MIGRATION AND MOBILITY
2,Austria,2018,LEGAL MIGRATION AND MOBILITY,ECONOMIC MIGRATION,New regulations that entered into force at the...,LEGAL MIGRATION AND MOBILITY
3,Austria,2018,LEGAL MIGRATION AND MOBILITY,FAMILY REUNIFICATION,No relevant changes were reported in 2018.,LEGAL MIGRATION AND MOBILITY
4,Austria,2018,INTERNATIONAL PROTECTION INCLUDING ASYLUM,Main,Several legislative acts relating to asylum an...,INTERNATIONAL PROTECTION AND ASYLUM
...,...,...,...,...,...,...
150,Armenia,2022,TRAFFICKING IN HUMAN BEINGS,NATIONAL STRATEGIC POLICY DEVELOPMENTS,The 6th National Programme on the Fight Agains...,TRAFFICKING IN HUMAN BEINGS
151,Armenia,2022,TRAFFICKING IN HUMAN BEINGS,IDENTIFICATION AND INFORMATION TO VICTIMS OF T...,"On 1 July 2022, the new Criminal Code and Crim...",TRAFFICKING IN HUMAN BEINGS
152,Armenia,2022,RETURN AND READMISSION,MAIN NATIONAL DEVELOPMENTS,The government of the Republic of Armenia has ...,RETURN AND READMISSION
153,Armenia,2022,RETURN AND READMISSION,COOPERATION WITH THIRD COUNTRIES,"During 2022, within the framework of the readm...",RETURN AND READMISSION
